In [ ]:
# Paths (added for the public repo). Paths below are relative to this notebook;
# override with the DATA_ROOT / RUNS_ROOT environment variables.
import os
DATA_ROOT = os.environ.get('DATA_ROOT', '../../data')   # data/blouse, data/aihub_official
RUNS_ROOT = os.environ.get('RUNS_ROOT', '../../runs')   # checkpoints / metrics are written here


# 폐의류 재활용 분류 — **B2-MS: Multi-Scale Texture Block (경우 A · 누수 없음)**

B1 에서 **질감 분기를 멀티스케일로** 한 가지만 바꾼 독립 ablation.
split·데이터·레시피는 B0·B1 과 완전히 동일 (원본 `_split` 공유, extra 없음).

**★ 변경점**: 단일 3×3 대신 미세(3×3) + 넓은영역(dilated 3×3) 병렬로 여러 크기의 오염·손상 질감 포착.
**비교법**: `baseline_B1_texture_clean/test_metrics.json` 의 macro_f1 대비 본 노트북 macro_f1.


## 0. 환경 설치

In [ ]:
# Colab 기본 환경에는 대부분 설치되어 있으나, FLOPs 측정용 thop 만 추가 설치
!pip -q install thop seaborn scikit-learn


## 1. Imports & 재현성 시드

In [ ]:
import os, json, random, time, math, copy
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

def seed_everything(seed: int = 0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # 속도 우선
    torch.backends.cudnn.benchmark = True

seed_everything(0)


# === [TRUNCATED IMAGE FIX] 잘린(손상) 이미지도 읽도록 허용 ===
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


## 2. 설정 (Config)

In [ ]:
@dataclass
class Config:
    data_root: str = f'{DATA_ROOT}/blouse'   # 원본 데이터 (extra 없음)
    output_dir: str = f'{RUNS_ROOT}/baseline_B2_multiscale_clean'

    main_classes: tuple = ('재사용', '재활용', '폐기')

    img_size: int = 224
    batch_size: int = 32          # OOM 시 16
    num_workers: int = 0
    epochs: int = 20
    lr: float = 1e-4
    weight_decay: float = 1e-3    # [B0 레시피] 과적합 완화
    label_smoothing: float = 0.05
    use_amp: bool = True

    # === [B0 oversample 레시피] ===
    dropout: float = 0.3
    early_stop_patience: int = 4
    imbalance_strategy: str = 'none'   # 오버샘플로 균형 → 보정 끔
    oversample: bool = True

    # texture block 토글 (B0=False, B1/B2=True)
    use_texture_block: bool = True
    texture_mid_ch: int = 32

    val_ratio: float = 0.15
    test_ratio: float = 0.15
    seed: int = 0

cfg = Config()
cfg.output_dir = f"{cfg.output_dir}_seed{cfg.seed}"   # seed별 폴더로 분리 저장
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

## 3. 데이터 준비 — 매니페스트 빌드

In [ ]:
# (로컬 실행) Google Drive 마운트 불필요 — 실행해도 아무 동작 안 합니다.
print('local run: drive mount skipped')


In [ ]:
import json
import pandas as pd
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

STATE_MAP = {'dispose': '폐기', 'recycle': '재활용', 'reusable': '재사용'}
PREFIX_MAP = {'DP': '폐기', 'RC': '재활용', 'RU': '재사용'}


def state_from_filename(name: str):
    pref = Path(name).stem.split('_')[0].upper()
    return PREFIX_MAP.get(pref)


def parse_label_json(json_path: Path, image_lookup: dict):
    """AI-Hub JSON -> dict 또는 None."""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        return None

    meta = data.get('meta_information', {})
    big  = meta.get('big_category', '').strip().lower()
    med  = meta.get('medium_category', 'unknown').strip().lower()
    fname = meta.get('file_name') or f'{json_path.stem}.jpg'

    main = STATE_MAP.get(big) or state_from_filename(fname)
    if main is None:
        return None

    img_path = image_lookup.get(fname) or image_lookup.get(Path(fname).stem)
    if img_path is None:
        return None

    ann = data.get('annotations', {})
    return {
        'file_path': str(img_path),
        'main': main,
        'sub':  med,
        'clothes_bbox':    [b['bbox'] for b in ann.get('clothes', [])],
        'pollution_bbox':  [b['bbox'] for b in ann.get('pollution', [])],
        'damage_bbox':     [b['bbox'] for b in ann.get('damage', [])],
        'attachment_bbox': [b['bbox'] for b in ann.get('attachment', [])],
        'caption': data.get('image_caption', {}).get('caption', ''),
    }


def build_manifest(data_root: str) -> pd.DataFrame:
    """data_root 아래 모든 JSON 을 훑어 매니페스트 DataFrame 생성 (재귀 탐색)."""
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"data_root not found: {data_root}")

    print('Indexing images...', flush=True)
    image_lookup = {}
    for p in data_root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            image_lookup[p.name] = p
            image_lookup.setdefault(p.stem, p)
    print(f'  {len(image_lookup)} image files indexed')

    print('Parsing JSON labels...', flush=True)
    rows, skipped = [], 0
    for jp in data_root.rglob('*.json'):
        r = parse_label_json(jp, image_lookup)
        if r is None:
            skipped += 1
            continue
        rows.append(r)
    df = pd.DataFrame(rows).drop_duplicates(subset='file_path').reset_index(drop=True)
    print(f'  parsed {len(df)} samples, skipped {skipped}')
    return df


df = build_manifest(cfg.data_root)
assert len(df) > 0, '매니페스트가 비었습니다. cfg.data_root 경로 확인 필요.'
display(df.drop(columns=[c for c in df.columns if c.endswith('_bbox')]).head())
print('\n[Main] 클래스 분포'); print(df['main'].value_counts())
print('\n오염 bbox 보유:', (df['pollution_bbox'].apply(len) > 0).sum(),
      '| 손상 bbox 보유:', (df['damage_bbox'].apply(len) > 0).sum())

manifest_path = Path(cfg.output_dir) / 'manifest.parquet'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
try:
    df.to_parquet(manifest_path); print('\nSaved manifest ->', manifest_path)
except Exception:
    manifest_path = Path(cfg.output_dir) / 'manifest.csv'
    df.to_csv(manifest_path, index=False); print('\nSaved manifest (csv) ->', manifest_path)


## 4. Train / Val / Test 분할 (Stratified, 원본만)

In [ ]:
# === [경우 A] 원본 데이터만 사용 (extra 없음) — 오버샘플로 균형만 맞추는 깨끗한 split ===
df = df[df['main'].isin(cfg.main_classes)].reset_index(drop=True)
main2idx = {c: i for i, c in enumerate(cfg.main_classes)}
df['main_idx'] = df['main'].map(main2idx)
SUB_CLASSES = tuple(df['sub'].value_counts().index.tolist())
sub2idx = {c: i for i, c in enumerate(SUB_CLASSES)}
df['sub_idx'] = df['sub'].map(sub2idx)
print(f'Discovered {len(SUB_CLASSES)} sub_classes: {SUB_CLASSES}')

import os as _os
# 원본 공유 split(_split). 있으면 재사용(B0/B1/B2 동일 split), 없으면 생성 후 저장.
_split_dir = f'{RUNS_ROOT}/_split'
_os.makedirs(_split_dir, exist_ok=True)
_paths = {k: _os.path.join(_split_dir, f'{k}_files.csv') for k in ['train', 'val', 'test']}

if all(_os.path.exists(p) for p in _paths.values()):
    sets = {k: set(pd.read_csv(p)['file_path']) for k, p in _paths.items()}
    train_df = df[df['file_path'].isin(sets['train'])].reset_index(drop=True)
    val_df   = df[df['file_path'].isin(sets['val'])].reset_index(drop=True)
    test_df  = df[df['file_path'].isin(sets['test'])].reset_index(drop=True)
    print('[split] 기존 _split 재사용 (B0/B1/B2 공유, 이전 clean 런과 동일)')
else:
    train_df, temp_df = train_test_split(
        df, test_size=cfg.val_ratio + cfg.test_ratio,
        stratify=df['main_idx'], random_state=cfg.seed)
    _vr = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    val_df, test_df = train_test_split(
        temp_df, test_size=1 - _vr, stratify=temp_df['main_idx'], random_state=cfg.seed)
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
    for k, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
        d[['file_path']].to_csv(_paths[k], index=False)
    print(f'[split] 신규 stratified 분할 저장 -> {_split_dir}')

# 누수 점검 — train 이 val·test 와 겹치면 안 됨
_lk = set(train_df['file_path']) & (set(val_df['file_path']) | set(test_df['file_path']))
assert not _lk, f"[누수] train 이 val/test 와 {len(_lk)}개 겹침!"
print(f'Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}  (누수 없음)')
print('train 클래스 분포(오버샘플 전):',
      {cfg.main_classes[k]: int(v) for k, v in
       train_df['main_idx'].value_counts().sort_index().items()})


## 5. Augmentation (+ RandomErasing)

In [ ]:
MEAN = [0.485, 0.456, 0.406]   # ImageNet 통계 (사전학습 정렬)
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((cfg.img_size + 32, cfg.img_size + 32)),
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(p=0.5),   # 의도적 배제
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    # [OVERSAMPLE] 오염/손상 가림 모사 + 복제 샘플 암기 방지
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


## 6. Dataset & DataLoader (+ 오버샘플링)

In [ ]:
from PIL import Image

class ClothingDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['file_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['main_idx'])

# === [OVERSAMPLE] 소수 클래스 오프라인 오버샘플링 (train 에만) ===
if cfg.oversample:
    target = train_df['main_idx'].value_counts().max()
    parts = []
    for _idx, grp in train_df.groupby('main_idx'):
        n_rep = target // len(grp)
        rem   = target - n_rep * len(grp)
        rep = pd.concat([grp] * n_rep +
                        [grp.sample(rem, random_state=cfg.seed)], ignore_index=True)
        parts.append(rep)
    train_df_bal = (pd.concat(parts, ignore_index=True)
                      .sample(frac=1, random_state=cfg.seed).reset_index(drop=True))
    print('오버샘플링 후 train 분포:',
          {cfg.main_classes[k]: int(v) for k, v in
           train_df_bal['main_idx'].value_counts().sort_index().items()})
else:
    train_df_bal = train_df

train_ds = ClothingDataset(train_df_bal, train_tf)
val_ds   = ClothingDataset(val_df,   eval_tf)
test_ds  = ClothingDataset(test_df,  eval_tf)

class_counts = train_df['main_idx'].value_counts().sort_index().values
class_weights = torch.tensor(class_counts.sum() / (len(class_counts) * class_counts),
                             dtype=torch.float32)
print('클래스 카운트(오버샘플 전):', dict(zip(cfg.main_classes, class_counts.tolist())))
print('(참고) Loss class_weights:', class_weights.tolist())

def _worker_init(worker_id):
    from PIL import ImageFile
    ImageFile.LOAD_TRUNCATED_IMAGES = True

# 불균형 보정: 오버샘플로 균형 맞췄으므로 'none'(shuffle) 권장
if cfg.imbalance_strategy == 'sampler':
    sample_weights = train_df_bal['main_idx'].map(
        {i: float(w) for i, w in enumerate(1.0 / class_counts)}).values
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                    replacement=True)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=cfg.num_workers, pin_memory=True,
                              worker_init_fn=_worker_init)
    print('불균형 보정: WeightedRandomSampler (Loss class_weight 미적용)')
else:
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True,
                              worker_init_fn=_worker_init)
    print(f'불균형 보정 전략: {cfg.imbalance_strategy} (shuffle 로더)')

val_loader  = DataLoader(val_ds,  batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)


## 7. 모델 — ResNet-50 **+ Multi-Scale Texture Block** (+ Dropout head)

In [ ]:
class MultiScaleTextureBlock(nn.Module):
    """[B2-MS] 멀티스케일 질감 분기 — 미세(3x3) + 넓은영역(dilated 3x3) 병렬."""
    def __init__(self, mid_ch: int = 32, out_ch: int = 64):
        super().__init__()
        self.fine   = nn.Conv2d(3, mid_ch, 3, stride=1, padding=1, bias=False)              # 작은 시야
        self.coarse = nn.Conv2d(3, mid_ch, 3, stride=1, padding=2, dilation=2, bias=False)  # 넓은 시야(5x5 상당)
        self.fuse_bn = nn.BatchNorm2d(mid_ch * 2)
        self.relu = nn.ReLU(inplace=True)
        self.down = nn.Sequential(
            nn.Conv2d(mid_ch * 2, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
    def forward(self, x):
        t = torch.cat([self.fine(x), self.coarse(x)], dim=1)
        t = self.relu(self.fuse_bn(t))
        return self.down(t)


class TextureResNet(nn.Module):
    """[B2-MS] Plain ResNet-50 + MultiScale Texture Block + Dropout head. 융합/백본/헤드는 B1 과 동일."""
    def __init__(self, num_main: int = 3, use_texture: bool = True,
                 texture_mid_ch: int = 32, pretrained: bool = True, dropout: float = 0.0):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=weights)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.use_texture = use_texture
        if use_texture:
            self.texture_block = MultiScaleTextureBlock(mid_ch=texture_mid_ch, out_ch=64)
            self.texture_alpha = nn.Parameter(torch.zeros(1))
        self.dropout = nn.Dropout(p=dropout)
        self.head_main = nn.Linear(in_feat, num_main)

    def forward(self, x):
        bb = self.backbone
        s = bb.conv1(x); s = bb.bn1(s); s = bb.relu(s); s = bb.maxpool(s)
        if self.use_texture:
            s = s + self.texture_alpha * self.texture_block(x)
        f = bb.layer1(s); f = bb.layer2(f); f = bb.layer3(f); f = bb.layer4(f)
        f = bb.avgpool(f); f = torch.flatten(f, 1)
        f = self.dropout(f)
        return self.head_main(f)


model = TextureResNet(num_main=len(cfg.main_classes), use_texture=cfg.use_texture_block,
                      texture_mid_ch=cfg.texture_mid_ch, dropout=cfg.dropout).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_extra  = sum(p.numel() for n, p in model.named_parameters()
               if n.startswith(('texture_block', 'texture_alpha')))
print(f'Trainable params: {n_params/1e6:.2f} M  '
      f'(MultiScale: {cfg.use_texture_block}, +{n_extra/1e3:.1f}K vs B0, Dropout p={cfg.dropout})')


## 8. 학습 루프 (AMP + Cosine + Early stopping)

In [ ]:
# === imbalance_strategy 에 맞춰 Loss class_weight 적용 여부 결정 ===
if cfg.imbalance_strategy == 'class_weight':
    _loss_weight = class_weights.to(DEVICE); print('CrossEntropyLoss: class_weight 적용')
else:
    _loss_weight = None; print('CrossEntropyLoss: class_weight 미적용')

criterion = nn.CrossEntropyLoss(weight=_loss_weight, label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=cfg.lr * 0.01)
scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total, correct = 0.0, 0, 0
    all_y, all_p = [], []
    for imgs, ys in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        ys   = ys.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=cfg.use_amp):
            logits = model(imgs)
            loss = criterion(logits, ys)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds == ys).sum().item(); total += imgs.size(0)
        all_y += ys.cpu().tolist(); all_p += preds.cpu().tolist()
    p, r, f1, _ = precision_recall_fscore_support(all_y, all_p, average='macro', zero_division=0)
    return total_loss/total, correct/total, p, r, f1


history = []
best_f1, best_state, best_epoch = -1, None, 0
ckpt = Path(cfg.output_dir) / 'best.pt'
epochs_no_improve = 0
for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc, _, _, tr_f1 = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va_loss, va_acc, va_p, va_r, va_f1 = run_epoch(val_loader, train=False)
    scheduler.step()
    history.append(dict(epoch=epoch, tr_loss=tr_loss, va_loss=va_loss,
                        tr_acc=tr_acc, va_acc=va_acc, va_f1=va_f1))
    print(f'[{epoch:02d}/{cfg.epochs}] loss {tr_loss:.3f}/{va_loss:.3f}  '
          f'acc {tr_acc:.3f}/{va_acc:.3f}  val_F1 {va_f1:.3f}  ({time.time()-t0:.1f}s)')
    if va_f1 > best_f1:
        best_f1 = va_f1; best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch; epochs_no_improve = 0
        torch.save({'model': best_state, 'cfg': asdict(cfg),
                    'val_f1': best_f1, 'epoch': epoch}, ckpt)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg.early_stop_patience:
            print(f'Early stopping: val_F1 {cfg.early_stop_patience}에폭 연속 개선 없음 '
                  f'(best epoch {best_epoch}, F1 {best_f1:.4f})')
            break

print(f'\nBest val F1-macro: {best_f1:.4f}  (epoch {best_epoch}, saved -> {ckpt})')
hist_df = pd.DataFrame(history)
hist_df.to_csv(Path(cfg.output_dir) / 'history.csv', index=False)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist_df.epoch, hist_df.tr_loss, label='train'); ax[0].plot(hist_df.epoch, hist_df.va_loss, label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist_df.epoch, hist_df.tr_acc, label='train_acc')
ax[1].plot(hist_df.epoch, hist_df.va_acc, label='val_acc')
ax[1].plot(hist_df.epoch, hist_df.va_f1,  label='val_F1', linestyle='--')
ax[1].set_title('Acc / F1'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig(Path(cfg.output_dir)/'curves.png', dpi=120); plt.show()


## 9. 테스트 평가 — Classification Report & Confusion Matrix

In [ ]:
ckpt_obj = torch.load(Path(cfg.output_dir)/'best.pt', map_location=DEVICE)
model.load_state_dict(ckpt_obj.get('model', ckpt_obj) if isinstance(ckpt_obj, dict) else ckpt_obj)
model.eval()

all_y, all_p, all_prob = [], [], []
with torch.no_grad(), torch.amp.autocast('cuda', enabled=cfg.use_amp):
    for imgs, ys in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        all_prob.append(torch.softmax(logits, 1).cpu().numpy())
        all_p += logits.argmax(1).cpu().tolist()
        all_y += ys.tolist()
all_prob = np.concatenate(all_prob)

print(classification_report(all_y, all_p, target_names=cfg.main_classes, digits=4))
acc = accuracy_score(all_y, all_p)
p, r, f1, _ = precision_recall_fscore_support(all_y, all_p, average='macro', zero_division=0)
pw, rw, f1w, _ = precision_recall_fscore_support(all_y, all_p, average='weighted', zero_division=0)
print(f'Accuracy        : {acc:.4f}')
print(f'Macro P/R/F1    : {p:.4f} / {r:.4f} / {f1:.4f}')
print(f'Weighted P/R/F1 : {pw:.4f} / {rw:.4f} / {f1w:.4f}')

metrics = dict(accuracy=acc, macro_f1=f1, weighted_f1=f1w,
               macro_precision=p, macro_recall=r,
               weighted_precision=pw, weighted_recall=rw)
with open(Path(cfg.output_dir)/'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
font_path = r"C:/Windows/Fonts/malgun.ttf"
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
print("Font exists:", os.path.exists(font_path), "| font:", plt.rcParams["font.family"])


In [ ]:
cm = confusion_matrix(all_y, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cfg.main_classes, yticklabels=cfg.main_classes, ax=ax[0])
ax[0].set_title('Confusion Matrix (count)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=cfg.main_classes, yticklabels=cfg.main_classes, ax=ax[1])
ax[1].set_title('Confusion Matrix (row-normalized)'); ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout(); plt.savefig(Path(cfg.output_dir)/'confusion_matrix.png', dpi=120); plt.show()


## 10. 효율성 지표 — Params / FLOPs / FPS

In [ ]:
from thop import profile
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=DEVICE)
with torch.no_grad():
    flops, _ = profile(model, inputs=(dummy,), verbose=False)
model.eval()
with torch.no_grad():
    for _ in range(10): _ = model(dummy)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time(); N = 100
    for _ in range(N): _ = model(dummy)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    fps = N / (time.time() - t0)
mem_mb = (torch.cuda.max_memory_allocated() / 1024**2) if DEVICE.type == 'cuda' else 0.0
eff = dict(params_M=round(n_params/1e6, 3), trainable_M=round(n_trainable/1e6, 3),
           gflops=round(flops/1e9, 3), fps_batch1=round(fps, 2),
           peak_gpu_mem_MB=round(mem_mb, 1))
print(json.dumps(eff, indent=2))
with open(Path(cfg.output_dir)/'efficiency.json', 'w') as f:
    json.dump(eff, f, indent=2)


## 11. Ablation 로드맵 (경우 A · 원본 데이터만, 누수 없음)

| Step | 모델 | 데이터/레시피 | 비교 기준 |
|---|---|---|---|
| B0 | Plain ResNet-50 | 원본만, `_split` 공유 | 기준점 macro_f1 |
| B1 | + Texture-aware Block | **B0 와 동일** | vs B0 |
| B2-MS | + Multi-Scale Texture | **B0/B1 과 동일** | vs B1 |

**핵심**: 셋이 split·데이터·레시피를 공유하므로 macro_f1 차이는 오로지 texture block 구조 차이.
**참고**: 점수는 0.8대로 보이겠지만 이것이 누수 없는 **진짜 성능**입니다. 추가로 올리려면
폐기 집중 augmentation·Focal loss·threshold 튜닝·TTA 등 정직한 기법을 얹으면 됩니다.

저장 파일: `best.pt`, `history.csv`, `curves.png`, `test_metrics.json`,
`confusion_matrix.png`, `efficiency.json` (모두 `cfg.output_dir` 아래).
